In [ ]:
img_file = "results/IMG_1247_mask.jpg"
# img_file = "cloth_data_gen/output/images/cloth_0019.png"

import torch
from ultralytics import YOLO

from models.yolo_cnn import EnhancedYoloKeypointNet, YoloBackbone, soft_argmax

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model
yolo11 = YOLO('yolo11l-seg.pt')  # Or yolo11m-seg.pt, yolo11x-seg.pt, etc.
backbone_seq = yolo11.model.model[:12]
backbone = YoloBackbone(backbone_seq, selected_indices=[0,1,2,3,4,5,6,7,8,9,10,11])
input_dummy = torch.randn(1, 3, 512, 512)
with torch.no_grad():
    feats = backbone(input_dummy)
print("Feature shapes:", [f.shape for f in feats])
in_channels_list = [f.shape[1] for f in feats]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
keypoint_net = EnhancedYoloKeypointNet(backbone, in_channels_list)
model = keypoint_net
for param in model.backbone.parameters():
    param.requires_grad = False
model = model.to(device)
compiled_model = torch.compile(model)
compiled_model.load_state_dict(torch.load('models/keypoint_model.pth', map_location=device))

In [ ]:
import cv2
import numpy as np

img = cv2.imread(img_file)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img = img
img = cv2.resize(img, (128, 128))  # Resize to match model
images = torch.Tensor([np.transpose(img, (2, 0, 1))]).to(device)
outputs = compiled_model(images)
coords = soft_argmax(outputs)

In [ ]:
import matplotlib.pyplot as plt

# render the predicted keypoints on the image
for img, kp in zip(images.cpu().numpy(), coords.cpu().detach().numpy()):
    img = np.transpose(img, (1, 2, 0))
    # Convert RGB to BGR for OpenCV
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    for i in range(kp.shape[0]):
        cv2.circle(img, (int(kp[i][0]), int(kp[i][1])), 1, (255,0,0), -1)

In [ ]:
# cv2.imshow('Predicted Keypoints', img)
# cv2.waitKey(0)
cv2.imwrite('results/predicted_keypoints.jpg', cv2.cvtColor(img, cv2.COLOR_RGB2BGR))